<a href="https://colab.research.google.com/github/k8peevey-sys/ds2002-fa26/blob/main/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
#Searching for pop in product_name
pop = cat['product_name'].str.lower().str.contains('pop')
pop_rows = cat[pop]

print("Number of Pop-Tarts:", len(pop_rows))
print(pop_rows)

Number of Pop-Tarts: 3
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
#Isolating just the pop tart products
pop_rows = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]
#Isolating just rows with strawberry
straw= (pop_rows['product_name'].str.lower().str.contains('straw', na=False))
straw_rows = pop_rows[straw]

#Printing all SKU with strawberry and all without
print("Pop-Tarts with Strawberry:", straw_rows[['sku', 'product_name']])
non_straw = pop_rows[~straw]
print("Pop-Tarts without Strawberry:", non_straw[['sku', 'product_name']])

Pop-Tarts with Strawberry:              sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
Pop-Tarts without Strawberry: Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** _None_

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
#Mapping every strawberry SKU to canonical SKU
sku_map={'PT-12': 'PT-12', "SKU#459812": 'SKU#459812', 'POPTART-STRAW': 'PT-12'}

#Create new coumn with canonicals
cat['canonical']= (cat['sku'].map(sku_map).fillna(cat['sku']))
print(cat[['sku', 'canonical']].head())

             sku   canonical
0          PT-12       PT-12
1     SKU#459812  SKU#459812
2  POPTART-STRAW       PT-12


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
# Categories pre-cleaning
print('Before norm:', cat['category'].value_counts(dropna=False))

# Categories post-cleaning
cat['category'] = (cat['category'].str.strip().str.title())
print('After norm:', cat['category'].value_counts(dropna=(False)))

#Distinct values
print('Distinct categories:', cat['category'].unique())

Before norm: category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64
After norm: category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64
Distinct categories: ['Breakfast Foods' 'Breakfast']


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
import re

def normalize_product_name(name):
  name = str(name).lower()
  name = re.sub(r'[^\w\s]', '', name)
  words = name.split()
  words.sort()
  return ' '.join(words)

cat['normalized_product_name'] = cat['product_name'].apply(normalize_product_name)

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
# TODO
rows_before = len(cat)

assert cat['canonical'].isnull().sum() == 0, "Nulls in canonical column"
rows_after = len(cat)

assert rows_before == rows_after, 'Row count changed after mapping'

print('Rows before:', rows_before)
print('Rows after:', rows_after)

Rows before: 3
Rows after: 3


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** Coca-Cola and Lay's chips both also appear fragmented. I would group them by standardizing the product names with all lowercase, no abbteviations and no punctuation and then grouping manually.

**b)** Consolidation becomes a business decision when there it is unclear whether or not something should be left as one thing or separated into distinct products. For example, some of the cereals have the same name but different packages. I would ask a marketing or branding team.